# recs_027 -- Catalog pooling x blend-weight grid on bge-small-en-v1.5 + vector-blend query

## Executive Summary

- `recs_025`'s catalog-side grid (best: `any_polarity__flat`, `blend_weight=0.3`) was found on
  USE + text-concat querying -- stale now that bge-small (`recs_024`) and vector-blend querying
  (`recs_026`) are both wins. This reruns the same grid on the actual pipeline being shipped.
- Fixed: `bge-small-en-v1.5`, vector-blend query at `w=0.5` (one of `recs_026`'s validated peak
  points, used as the sanity-check anchor).
- Result: **All 24 cells beat `two_tower_v1`** (vs. zero out of 24 in `recs_025`'s USE grid) --
  the vector-blend query switch alone was enough. Catalog tuning adds a further real gain on top:
  best cell `any_polarity__log_weighted` / `blend_weight=0.05` -> 0.532/0.517/0.514, beating
  `recs_026`'s untuned baseline (0.530/0.509/0.513). `log_weighted` pooling wins here -- a
  reversal from `recs_025`, where it lost under USE.

## Business Context

`recs_026` already beats `two_tower_v1` using the shipped catalog defaults
(`any_polarity__flat`, `blend_weight=0.1`) -- untuned. This checks whether catalog-side tuning
adds anything further on top of the vector-blend query win, or whether the shipped defaults were
already good enough.

## Research Question

Holding bge-small + vector-blend querying (`w=0.5`) fixed, does catalog pooling variant or
blend weight move the numbers further, the way blend weight did (barely) under USE in
`recs_025`?

## Hypothesis

Catalog-side blend weight was a tiny lever under USE (`recs_025`, +0.001-0.004). Query-side
blend weight was a huge lever under bge-small (`recs_026`, +0.014-0.016 over text-concat).
Prediction: catalog-side tuning stays a minor lever here too -- the two axes aren't obviously
coupled, so no strong reason to expect the query-side result to repeat on the catalog side.

**Result: wrong.** Catalog tuning gave a real gain here (+0.008 Recall@K Slice A over the
untuned baseline), bigger than `recs_025`'s USE-side effect (+0.004). And `log_weighted`
pooling flipped from losing (USE) to winning (bge-small) -- the two axes aren't independent of
embedder choice the way the hypothesis assumed.

## Definitions

| Term | Meaning |
|---|---|
| Catalog pooling/blend | Same as `recs_025`: 4 variants (`any_polarity`/`recommended_only` x `flat`/`log_weighted`), blended with description at `blend_weight`. |
| Vector-blend query | Same as `recs_026`: `normalize((1-w)*review_vec + w*description_vec)`, held fixed at `w=0.5` here. |

## Data Sources

| Source | Role |
|---|---|
| `artifacts/recs/embeddings/game_chunks/default/game_review_chunks.parquet` | Stage 1 chunk table, catalog side. |
| `artifacts/recs/offline_eval/runs/rag_v1/eval_offline_examples.jsonl` | Same 12,500-example cohort used throughout this series. |
| `data/processed/steam_reviews_cleaned_english_val_norm.parquet` | Val split -- recovers query review text. |

## Design / Process

1. Embed all chunks + queries with bge-small once (query side gets the BGE prefix).
2. For each of 4 pooling variants x 6 catalog blend weights: build the catalog, score against
   query vectors blended at fixed `w=0.5`.
3. Sanity check: `any_polarity__flat` / catalog `blend_weight=0.1` should reproduce `recs_026`'s
   real `w=0.5` number (0.530 / 0.509 / 0.513).
4. Compare the full grid against `two_tower_v1` and `recs_026`'s baseline.

## Evaluation Outputs / Artifacts

| Artifact | Description |
|---|---|
| Comparison grid (this notebook) | Hit@100/Recall@100, overall and by slice, for 4 pooling variants x 6 catalog blend weights. |

## Notebook Roadmap

1. Setup
2. Load chunk table + eval cohort, recover query text
3. Embed catalog chunks + query components with bge-small (one pass)
4. Pool + blend + score the full grid
5. Comparison table + findings

# Analysis

## Setup

In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

def _find_repo_root(start: Path) -> Path:
    p = start.resolve()
    while not (p / "pyproject.toml").is_file():
        if p.parent == p:
            raise RuntimeError("Could not find repo root (pyproject.toml not found).")
        p = p.parent
    return p

REPO_ROOT = _find_repo_root(Path.cwd())
import sys
sys.path.insert(0, str(REPO_ROOT / "src"))

from steam_review_ml.recommender.math_utils import l2_normalize
from steam_review_ml.evaluation.retrieval_offline_eval import hit_rate_at_k, precision_at_k, recall_at_k

RUN_DIR = REPO_ROOT / "artifacts" / "recs" / "offline_eval" / "runs" / "rag_v1"
CHUNKS_PATH = REPO_ROOT / "artifacts" / "recs" / "embeddings" / "game_chunks" / "default" / "game_review_chunks.parquet"
VAL_SPLIT_PATH = REPO_ROOT / "data" / "processed" / "steam_reviews_cleaned_english_val_norm.parquet"

USER_COL = "author.steamid"
K_RETRIEVAL = 100
CATALOG_BLEND_WEIGHTS = [0.0, 0.05, 0.1, 0.2, 0.3, 0.5]
QUERY_VECTOR_BLEND_WEIGHT = 0.5
POLARITY_ARMS = ("any_polarity", "recommended_only")
WEIGHTING_ARMS = ("flat", "log_weighted")
BGE_QUERY_PREFIX = "Represent this sentence for searching relevant passages: "

pd.options.display.max_colwidth = 60
print(f"REPO_ROOT={REPO_ROOT}")

REPO_ROOT=/home/ryanr/workspace/steam_recommendations


## Load Stage 1 Chunk Table + Eval Cohort, Recover Query Text

In [2]:
chunks_df = pd.read_parquet(CHUNKS_PATH)
review_chunks = chunks_df[chunks_df["chunk_type"] == "review"].reset_index(drop=True)
description_by_app: dict[int, str] = dict(
    zip(
        chunks_df.loc[chunks_df["chunk_type"] == "description", "app_id"],
        chunks_df.loc[chunks_df["chunk_type"] == "description", "text"],
    )
)
print(f"chunk rows: {len(chunks_df):,} (review={len(review_chunks):,}, description={len(description_by_app):,})")

examples = []
with open(RUN_DIR / "eval_offline_examples.jsonl") as f:
    for line in f:
        rec = json.loads(line)
        if rec["method"] != "rag_chunk_v1_query_plus_desc":
            continue
        examples.append(
            {
                "ex_idx": rec["ex_idx"],
                "user_id": rec["user_id"],
                "query_app_id": rec["query_app_id"],
                "positives": set(json.loads(rec["validation_positive_app_ids_json"])),
                "n_eval_targets": rec["n_eval_targets"],
                "slice_name": rec["slice_name"],
            }
        )
examples_df = pd.DataFrame(examples)
print(f"eval cohort rows: {len(examples_df):,}")

val_df = pd.read_parquet(VAL_SPLIT_PATH, columns=[USER_COL, "app_id", "review"])
val_df["_key"] = val_df[USER_COL].astype(str) + "::" + val_df["app_id"].astype(str)
review_text_by_key = dict(zip(val_df["_key"], val_df["review"]))
examples_df["query_review_text"] = examples_df.apply(
    lambda r: review_text_by_key[f"{r['user_id']}::{r['query_app_id']}"], axis=1
)
display(examples_df[["ex_idx", "query_app_id", "n_eval_targets", "slice_name"]].head(3))

chunk rows: 16,010 (review=15,695, description=315)
eval cohort rows: 12,500


   ex_idx  query_app_id  n_eval_targets             slice_name0       0        812140               1  slice_b_single_target1       1        485510               1  slice_b_single_target2       2        646570               2   slice_a_multi_target

## Embed Catalog Chunks + Query Components With bge-small (One Pass)

In [3]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("BAAI/bge-small-en-v1.5", device="cuda")


def encode(texts: list[str], *, prefix: str = "") -> np.ndarray:
    prefixed = [prefix + t for t in texts]
    emb = np.asarray(model.encode(prefixed, batch_size=128, show_progress_bar=True))
    return np.stack([l2_normalize(row) for row in emb], axis=0)


# Catalog (passage) side: no query prefix.
catalog_review_vecs = encode(review_chunks["text"].tolist())
desc_app_ids = list(description_by_app.keys())
catalog_desc_by_app = {a: v for a, v in zip(desc_app_ids, encode([description_by_app[a] for a in desc_app_ids]))}

# Query side: BGE prefix on both components, blended at the fixed weight from recs_026.
query_review_vecs = encode(examples_df["query_review_text"].tolist(), prefix=BGE_QUERY_PREFIX)
query_desc_by_app = {a: v for a, v in zip(desc_app_ids, encode([description_by_app[a] for a in desc_app_ids], prefix=BGE_QUERY_PREFIX))}

query_vecs = []
for i, ex in enumerate(examples_df.itertuples(index=False)):
    desc_vec = query_desc_by_app.get(int(ex.query_app_id))
    if desc_vec is None:
        query_vecs.append(query_review_vecs[i])
    else:
        w = QUERY_VECTOR_BLEND_WEIGHT
        query_vecs.append(l2_normalize((1.0 - w) * query_review_vecs[i] + w * desc_vec))
query_vecs = np.stack(query_vecs, axis=0)
print(f"catalog_review_vecs: {catalog_review_vecs.shape}, query_vecs: {query_vecs.shape}")

catalog_review_vecs: (15695, 384), query_vecs: (12500, 384)


## Pool + Blend + Score the Full Grid

In [4]:
def pool_review_variant(review_df: pd.DataFrame, review_vecs: np.ndarray, *, polarity: str, weighting: str) -> dict[int, np.ndarray]:
    sub = review_df[review_df["recommended"] == 1] if polarity == "recommended_only" else review_df
    out: dict[int, np.ndarray] = {}
    for app_id, g in sub.groupby("app_id"):
        idx = g.index.to_numpy()
        vecs = review_vecs[idx]
        if weighting == "log_weighted":
            w = np.log1p(g["votes_helpful"].to_numpy(dtype=np.float64))
            if w.sum() <= 0:
                w = np.ones(len(g), dtype=np.float64)
        else:
            w = np.ones(len(g), dtype=np.float64)
        w = w / w.sum()
        out[int(app_id)] = l2_normalize((vecs * w[:, None]).sum(axis=0))
    return out


def blend_catalog(pooled_by_app: dict[int, np.ndarray], desc_by_app: dict[int, np.ndarray], blend_weight: float) -> tuple[np.ndarray, np.ndarray]:
    app_ids_out, vecs_out = [], []
    for app_id, desc_vec in desc_by_app.items():
        pooled = pooled_by_app.get(app_id)
        if pooled is None:
            continue
        blended = l2_normalize((1.0 - blend_weight) * pooled + blend_weight * desc_vec)
        app_ids_out.append(app_id)
        vecs_out.append(blended)
    return np.asarray(app_ids_out, dtype=np.int64), np.stack(vecs_out, axis=0)


def score_examples(app_ids: np.ndarray, catalog_matrix: np.ndarray) -> pd.DataFrame:
    app_to_row = {int(a): i for i, a in enumerate(app_ids)}
    rows = []
    for i, ex in enumerate(examples_df.itertuples(index=False)):
        scores = catalog_matrix @ query_vecs[i]
        self_row = app_to_row.get(int(ex.query_app_id))
        if self_row is not None:
            scores[self_row] = -np.inf
        ranked_rows = np.argsort(-scores)[:K_RETRIEVAL]
        rows.append(
            {
                "slice_name": ex.slice_name,
                "Hit@K": hit_rate_at_k(ranked_rows, ex.positives, K_RETRIEVAL, app_ids),
                "Recall@K": recall_at_k(ranked_rows, ex.positives, K_RETRIEVAL, app_ids),
            }
        )
    return pd.DataFrame(rows)


def summarize(df: pd.DataFrame) -> dict:
    by_slice = df.groupby("slice_name")[["Hit@K", "Recall@K"]].mean()
    return {
        "overall_hit": df["Hit@K"].mean(),
        "slice_a_recall": by_slice.loc["slice_a_multi_target", "Recall@K"] if "slice_a_multi_target" in by_slice.index else float("nan"),
        "slice_b_hit": by_slice.loc["slice_b_single_target", "Hit@K"] if "slice_b_single_target" in by_slice.index else float("nan"),
    }


grid_rows = []
for polarity in POLARITY_ARMS:
    for weighting in WEIGHTING_ARMS:
        variant = f"{polarity}__{weighting}"
        pooled_by_app = pool_review_variant(review_chunks, catalog_review_vecs, polarity=polarity, weighting=weighting)
        for bw in CATALOG_BLEND_WEIGHTS:
            app_ids, catalog_matrix = blend_catalog(pooled_by_app, catalog_desc_by_app, bw)
            s = summarize(score_examples(app_ids, catalog_matrix))
            grid_rows.append(
                {
                    "pooling_variant": variant,
                    "catalog_blend_weight": bw,
                    "n_games": len(app_ids),
                    "Hit@K (overall)": s["overall_hit"],
                    "Recall@K (Slice A)": s["slice_a_recall"],
                    "Hit@K (Slice B)": s["slice_b_hit"],
                }
            )
        print(f"scored variant={variant}")

grid_df = pd.DataFrame(grid_rows)
print(f"grid rows: {len(grid_df)}")

scored variant=any_polarity__flat
scored variant=any_polarity__log_weighted
scored variant=recommended_only__flat
scored variant=recommended_only__log_weighted
grid rows: 24


## Sanity Check + Comparison Grid

In [5]:
sanity = grid_df[(grid_df["pooling_variant"] == "any_polarity__flat") & (grid_df["catalog_blend_weight"] == 0.1)].iloc[0]
print("This notebook's any_polarity__flat / catalog_blend_weight=0.1 cell:")
print(f"  Hit@K={sanity['Hit@K (overall)']:.3f}  Recall@K(A)={sanity['Recall@K (Slice A)']:.3f}  Hit@K(B)={sanity['Hit@K (Slice B)']:.3f}")
print("recs_026's real vector_blend w=0.5 number (same catalog config):")
print("  Hit@K=0.530  Recall@K(A)=0.509  Hit@K(B)=0.513")

print()
print("two_tower_v1 bar: Hit@K=0.512  Recall@K(A)=0.460  Hit@K(B)=0.496")
grid_df["beats_two_tower_all_3"] = (
    (grid_df["Hit@K (overall)"] > 0.512) & (grid_df["Recall@K (Slice A)"] > 0.460) & (grid_df["Hit@K (Slice B)"] > 0.496)
)
display(grid_df.sort_values("Recall@K (Slice A)", ascending=False).round(3))

This notebook's any_polarity__flat / catalog_blend_weight=0.1 cell:
  Hit@K=0.530  Recall@K(A)=0.509  Hit@K(B)=0.513
recs_026's real vector_blend w=0.5 number (same catalog config):
  Hit@K=0.530  Recall@K(A)=0.509  Hit@K(B)=0.513

two_tower_v1 bar: Hit@K=0.512  Recall@K(A)=0.460  Hit@K(B)=0.496


                   pooling_variant  catalog_blend_weight  n_games  Hit@K (overall)  Recall@K (Slice A)  Hit@K (Slice B)  beats_two_tower_all_37       any_polarity__log_weighted                  0.05      315            0.532               0.517            0.514                   True21  recommended_only__log_weighted                  0.20      314            0.530               0.514            0.512                   True15          recommended_only__flat                  0.20      314            0.530               0.513            0.512                   True1               any_polarity__flat                  0.05      315            0.530               0.512            0.512                   True18  recommended_only__log_weighted                  0.00      314            0.527               0.511            0.509                   True0               any_polarity__flat                  0.00      315            0.528               0.510            0.510                   True20  re

## Key Findings

| pooling_variant | catalog_blend_weight | Hit@K (overall) | Recall@K (Slice A) | Hit@K (Slice B) |
|---|---|---|---|---|
| `two_tower_v1` (bar) | -- | 0.512 | 0.460 | 0.496 |
| `recs_026` baseline (untuned catalog) | 0.1 | 0.530 | 0.509 | 0.513 |
| **`any_polarity__log_weighted` (best)** | **0.05** | **0.532** | **0.517** | **0.514** |
| `recommended_only__log_weighted` | 0.20 | 0.530 | 0.514 | 0.512 |
| `recommended_only__flat` | 0.20 | 0.530 | 0.513 | 0.512 |

- **Sanity check exact match** at the untuned catalog config, confirming this notebook's scoring
  matches `recs_026`.
- **Every one of the 24 cells beats `two_tower_v1`** -- unlike `recs_025` (0/24 on USE), the
  vector-blend query switch alone clears the bar regardless of catalog config.
- **Catalog tuning still adds a real, if modest, gain on top**: +0.008 Recall@K Slice A over the
  untuned default, roughly double `recs_025`'s USE-side effect (+0.004).
- **`log_weighted` reverses from `recs_025`**: it lost to `flat` under USE, wins here under
  bge-small + vector-blend querying. Confirms the `recs_026` finding -- pipeline optima aren't
  fixed properties of the data, they interact with embedder and query-construction choice.

## Recommendation / Next Steps

- **Ship `any_polarity__log_weighted` + `catalog_blend_weight=0.05` + vector-blend query
  (`w=0.5`) + `bge-small-en-v1.5`** as the new full RAG retrieval config -- best result across
  the entire ablation series, and every cell tested here already clears `two_tower_v1`.
- Needs promotion through the real pipeline before finalizing: Stage 2 rebuild with
  `log_weighted` pooling and `blend_weight=0.05` (currently `any_polarity__flat`/`0.1`), Stage 3
  retriever wired for vector-blend query construction (currently text-concat), then a real
  `recs_job_eval_offline.py` run to confirm.
- Re-run the franchise/marketing qualitative check (`recs_023`) once promoted -- untested against
  this config.